# RetailIQ - Data Cleaning

Cleans the raw synthetic retail sales dataset: fixes types, handles missing values & duplicates, removes/caps outliers, and engineers date-based features. Mirrors `analytics/cleaning.py`.

## 1. Load raw data

In [1]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv("../data/raw/retail_sales_raw.csv")
print(df.shape)
df.head()

(60900, 21)


,Order_ID,Date,Customer_ID,Customer_Name,City,State,Region,Store_ID,Store_Name,Product_ID,...,Category,Sub_Category,Quantity,Unit_Price,Discount,Sales,Cost,Profit,Supplier,Payment_Method
0,ORD017504,2024-05-05,CUST00396,Karan Agarwal,Ahmedabad,Gujarat,West,ST009,RetailIQ Hyderabad Hitech City,PRD00441,...,Books & Stationery,Office Supplies,1,1184.75,0.00,1184.75,836.12,348.63,OfficeMart Supplies,Cash on Delivery
1,ORD046116,2025-11-04,CUST03696,Vihaan Gupta,Warangal,Telangana,South,ST007,RetailIQ Kolkata Park St,PRD00244,...,Home & Kitchen,Storage,1,12888.61,0.15,10955.32,7594.54,3360.78,KitchenCraft Suppliers,UPI
2,ORD059784,2024-04-14,CUST00665,Tanya Das,Hyderabad,Telangana,South,ST002,RetailIQ Noida Hub,PRD00278,...,Grocery,Beverages,3,587.75,0.00,1763.25,1204.56,558.69,DailyNeeds Distributors,Debit Card
3,ORD011462,2024-07-04,CUST01154,Kabir Joshi,Noida,Uttar Pradesh,North,ST007,RetailIQ Kolkata Park St,PRD00449,...,Books & Stationery,Office Supplies,1,168.57,0.00,168.57,115.16,53.41,PageTurner Distributors,UPI
4,ORD050030,2024-10-13,CUST00655,Rahul Gupta,Chennai,Tamil Nadu,South,ST010,RetailIQ Patna Boring Road,PRD00265,...,Grocery,Snacks,1,595.62,0.00,595.62,454.28,141.34,FreshMart Supply Chain,Credit Card


## 2. Initial data quality check

In [2]:
print(df.dtypes)
print("\nMissing values:\n", df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())

Order_ID              str
Date                  str
Customer_ID           str
Customer_Name         str
City                  str
State                 str
Region                str
Store_ID              str
Store_Name            str
Product_ID            str
Product_Name          str
Category              str
Sub_Category          str
Quantity              str
Unit_Price        float64
Discount          float64
Sales             float64
Cost              float64
Profit            float64
Supplier              str
Payment_Method        str
dtype: object

Missing values:
 Order_ID             0
Date                 0
Customer_ID          0
Customer_Name     1215
City               606
State                0
Region               0
Store_ID             0
Store_Name           0
Product_ID           0
Product_Name         0
Category             0
Sub_Category         0
Quantity             0
Unit_Price           0
Discount           914
Sales                0
Cost                 0
Profit  

## 3. Fix Quantity column (mixed types like '2 units')

In [3]:
def clean_quantity(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, str):
        match = re.search(r"-?\d+", value)
        return int(match.group()) if match else np.nan
    return int(value)

df["Quantity"] = df["Quantity"].apply(clean_quantity)
df["Quantity"] = df["Quantity"].abs()  # fix sign-entry errors
df["Quantity"].describe()

count    60900.000000
mean         1.469737
std          0.879692
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max          5.000000
Name: Quantity, dtype: float64

## 4. Parse & validate dates

In [4]:
def parse_date(value):
    try:
        return pd.to_datetime(value, format="%Y-%m-%d")
    except (ValueError, TypeError):
        return pd.NaT

df["Date"] = df["Date"].apply(parse_date)
print("Invalid dates dropped:", df["Date"].isna().sum())
df = df[df["Date"].notna()].copy()

Invalid dates dropped: 304


## 5. Remove duplicates

In [5]:
before = len(df)
df = df.drop_duplicates()
print(f"Removed {before - len(df)} duplicate rows")

Removed 886 duplicate rows


## 6. Handle missing values

In [6]:
df["Customer_Name"] = df["Customer_Name"].fillna("Unknown Customer")
df["Discount"] = df["Discount"].fillna(0.0)
df["Payment_Method"] = df["Payment_Method"].fillna("Unknown")
df["City"] = df["City"].fillna("Unknown")
df.isna().sum()

Order_ID          0
Date              0
Customer_ID       0
Customer_Name     0
City              0
State             0
Region            0
Store_ID          0
Store_Name        0
Product_ID        0
Product_Name      0
Category          0
Sub_Category      0
Quantity          0
Unit_Price        0
Discount          0
Sales             0
Cost              0
Profit            0
Supplier          0
Payment_Method    0
dtype: int64

## 7. Outlier detection (IQR method, per category)

In [7]:
df["Sales_recalc"] = (df["Quantity"] * df["Unit_Price"] * (1 - df["Discount"])).round(2)

def cap_outliers(group):
    q1, q3 = group.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = max(0, q1 - 1.5*iqr), q3 + 1.5*iqr
    return group.clip(lower, upper)

df["Sales"] = df.groupby("Category")["Sales_recalc"].transform(cap_outliers)
df[["Sales_recalc", "Sales"]].describe()

,Sales_recalc,Sales
count,59710.000000,59710.000000
mean,14910.084036,13867.828934
std,29070.224968,24188.678505
min,17.830000,17.830000
25%,1325.160000,1325.160000
50%,3984.520000,3927.100000
75%,14832.610000,14768.360000
max,417956.900000,145251.220000


## 8. Feature engineering

In [8]:
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Quarter"] = df["Date"].dt.quarter
df["Weekday"] = df["Date"].dt.day_name()
df["Cost"] = (df["Quantity"] * df["Unit_Price"] * 0.68).round(2)
df["Profit"] = (df["Sales"] - df["Cost"]).round(2)
df["Margin_Pct"] = (df["Profit"] / df["Sales"] * 100).round(2)
df.head()

,Order_ID,Date,Customer_ID,Customer_Name,City,State,Region,Store_ID,Store_Name,Product_ID,...,Cost,Profit,Supplier,Payment_Method,Sales_recalc,Year,Month,Quarter,Weekday,Margin_Pct
0,ORD017504,2024-05-05,CUST00396,Karan Agarwal,Ahmedabad,Gujarat,West,ST009,RetailIQ Hyderabad Hitech City,PRD00441,...,805.63,379.12,OfficeMart Supplies,Cash on Delivery,1184.75,2024,5,2,Sunday,32.0
1,ORD046116,2025-11-04,CUST03696,Vihaan Gupta,Warangal,Telangana,South,ST007,RetailIQ Kolkata Park St,PRD00244,...,8764.25,2191.07,KitchenCraft Suppliers,UPI,10955.32,2025,11,4,Tuesday,20.0
2,ORD059784,2024-04-14,CUST00665,Tanya Das,Hyderabad,Telangana,South,ST002,RetailIQ Noida Hub,PRD00278,...,1199.01,564.24,DailyNeeds Distributors,Debit Card,1763.25,2024,4,2,Sunday,32.0
3,ORD011462,2024-07-04,CUST01154,Kabir Joshi,Noida,Uttar Pradesh,North,ST007,RetailIQ Kolkata Park St,PRD00449,...,114.63,53.94,PageTurner Distributors,UPI,168.57,2024,7,3,Thursday,32.0
4,ORD050030,2024-10-13,CUST00655,Rahul Gupta,Chennai,Tamil Nadu,South,ST010,RetailIQ Patna Boring Road,PRD00265,...,405.02,190.60,FreshMart Supply Chain,Credit Card,595.62,2024,10,4,Sunday,32.0


## 9. Export cleaned dataset

(For the full production version with all edge cases, see `analytics/cleaning.py`)

In [9]:
df.to_csv("../data/cleaned/retail_sales_cleaned_notebook.csv", index=False)
print("Saved cleaned dataset:", df.shape)

Saved cleaned dataset: (59710, 27)
